In [ ]:
import sys
sys.path.append('..')

from pathlib import Path
from datasets import load_dataset
import pandas as pd
import torch
from sentence_transformers.sentence_transformer.evaluation import InformationRetrievalEvaluator
from sentence_transformers.sparse_encoder.evaluation import SparseInformationRetrievalEvaluator
from sentence_transformers import (
    SentenceTransformer,
    SparseEncoder,
    CrossEncoder
)

from hybrid_retrieval.util.io import read_json
from hybrid_retrieval.eval import WeightedReciprocalRankFusionEvaluator

pd.set_option('display.max_columns', None)

In [5]:
from sentence_transformers.sparse_encoder.losses import SparseMultipleNegativesRankingLoss, CachedSpladeLoss

loss = SparseMultipleNegativesRankingLoss(model=None)
loss = CachedSpladeLoss(
    mini_batch_size=8,
    model=None,
    loss=loss,
    query_regularizer_weight=3e-3,
    # use_document_regularizer_only=True,
    document_regularizer_weight=3e-3,
)

In [ ]:
dataset_dir = Path('../data/dataset')

inv_qrels = read_json(dataset_dir / 'inv_qrels.json')
passages_splits = read_json(dataset_dir / 'passages_splits2.json')
passages = read_json(dataset_dir / 'passages.json')
queries = read_json(dataset_dir / 'queries.json')

queries = {id:q for q, id in zip(queries['query'], queries['id'])}
queries_to_id = {text:qid for qid, text in queries.items()}
passages = {id:p for p, id in zip(passages['passage'], passages['id'])}
passage_to_id = {text:pid for pid, text in passages.items()}


qrels_split = lambda split : {
    qid:pid 
    for pid, qids in zip(*inv_qrels.values()) 
    for qid in qids
    if pid in passages_splits[split]
}

qrels_eval = qrels_split('test')
qrels_eval_ = {qid: [pid] for qid, pid in qrels_eval.items()}

queries_eval = {qid:queries[qid] for qid in qrels_eval}

s = f'''
qrels_eval: {len(qrels_eval)}
queries_eval: {len(queries_eval)}
'''

In [ ]:
model_name = "deepvk/USER-bge-m3"

model = SentenceTransformer(
    model_name, 
    device='cuda' if torch.cuda.is_available() else 'cpu'
)
model.eval()

In [ ]:
ir_evaluator = InformationRetrievalEvaluator(
    queries=queries_eval,
    corpus=passages,
    relevant_docs=qrels_eval_,
    name="ir_evaluator",
    map_at_k=[20],
    show_progress_bar=True,
    write_predictions=True
)

results = ir_evaluator(model)

In [ ]:
# results  # до fine-tuning

# {'ir_evaluator_cosine_accuracy@1': 0.6071428571428571,
#  'ir_evaluator_cosine_accuracy@3': 0.8169642857142857,
#  'ir_evaluator_cosine_accuracy@5': 0.8705357142857143,
#  'ir_evaluator_cosine_accuracy@10': 0.90625,
#  'ir_evaluator_cosine_precision@1': 0.6071428571428571,
#  'ir_evaluator_cosine_precision@3': 0.27232142857142855,
#  'ir_evaluator_cosine_precision@5': 0.17410714285714285,
#  'ir_evaluator_cosine_precision@10': 0.090625,
#  'ir_evaluator_cosine_recall@1': 0.6071428571428571,
#  'ir_evaluator_cosine_recall@3': 0.8169642857142857,
#  'ir_evaluator_cosine_recall@5': 0.8705357142857143,
#  'ir_evaluator_cosine_recall@10': 0.90625,
#  'ir_evaluator_cosine_ndcg@10': 0.7627457729482716,
#  'ir_evaluator_cosine_mrr@10': 0.715826955782313,
#  'ir_evaluator_cosine_map@20': 0.7194184819906982}

#### SPLADE

In [ ]:
model_sparse_splade = SparseEncoder(
    'i1j/retriever-sparse-splade',
    device='cuda' if torch.cuda.is_available() else 'cpu'
)

In [ ]:
output = model_sparse_splade.encode(['блокировочный путь'])
model_sparse_splade.decode(output, top_k=20)

[[('блок', 1.992846131324768),
  ('путь', 1.9476943016052246),
  ('пути', 1.6240466833114624),
  ('path', 1.0806922912597656),
  ('##иро', 1.065852165222168),
  ('##во', 1.0124300718307495),
  ('طريق', 1.0055228471755981),
  ('##чныи', 0.9722108840942383),
  ('шлях', 0.9588326215744019),
  ('пут', 0.7444075345993042),
  ('block', 0.7286161184310913),
  ('##чная', 0.7059526443481445),
  ('via', 0.6856067776679993),
  ('##чные', 0.6052811741828918),
  ('ruta', 0.586514413356781),
  ('الطريق', 0.5784600973129272),
  ('##чную', 0.5688701868057251),
  ('##вои', 0.5648748874664307),
  ('путем', 0.5452020168304443),
  ('##очныи', 0.5423678159713745)]]

In [ ]:
irs_evaluator = SparseInformationRetrievalEvaluator(
    queries=queries_eval,
    corpus=passages,
    relevant_docs=qrels_eval_,
    name="irs_evaluator",
    map_at_k=[20, 100],
    write_predictions=True
)

results = irs_evaluator(model_sparse_splade, output_path="../result/eval/sparse_splade")

In [ ]:
results

{'irs_evaluator_dot_accuracy@1': 0.78125,
 'irs_evaluator_dot_accuracy@3': 0.9241071428571429,
 'irs_evaluator_dot_accuracy@5': 0.9553571428571429,
 'irs_evaluator_dot_accuracy@10': 0.9866071428571429,
 'irs_evaluator_dot_precision@1': 0.78125,
 'irs_evaluator_dot_precision@3': 0.3080357142857143,
 'irs_evaluator_dot_precision@5': 0.1910714285714286,
 'irs_evaluator_dot_precision@10': 0.0986607142857143,
 'irs_evaluator_dot_recall@1': 0.78125,
 'irs_evaluator_dot_recall@3': 0.9241071428571429,
 'irs_evaluator_dot_recall@5': 0.9553571428571429,
 'irs_evaluator_dot_recall@10': 0.9866071428571429,
 'irs_evaluator_dot_ndcg@10': 0.8855710842974792,
 'irs_evaluator_dot_mrr@10': 0.8528380102040817,
 'irs_evaluator_dot_map@20': 0.8531356292517006,
 'irs_evaluator_dot_map@100': 0.8534382086167801,
 'irs_evaluator_query_active_dims': 215.48214721679688,
 'irs_evaluator_query_sparsity_ratio': 0.9979648263846769,
 'irs_evaluator_corpus_active_dims': 361.7178649902344,
 'irs_evaluator_corpus_sparsi

#
#### RRF

In [ ]:
# model_name = 'i1j/retriever'

# model = SentenceTransformer(
#     model_name, 
#     device='cuda' if torch.cuda.is_available() else 'cpu'
# )

# qrels_eval_ = {qid: [pid] for qid, pid in qrels_eval.items()}

# ir_evaluator = InformationRetrievalEvaluator(
#     queries=queries_eval,
#     corpus=passages,
#     relevant_docs=qrels_eval_,
#     name="ir_evaluator",
#     map_at_k=[100],
#     show_progress_bar=True,
#     write_predictions=True
# )

# results = ir_evaluator(model, output_path=f"eval/dense")

In [5]:
dense_pred_path = f"../result/eval/dense/Information-Retrieval_evaluation_ir_evaluator_predictions_cosine.jsonl"
dense_metrics_path = f"../result/eval/dense/Information-Retrieval_evaluation_ir_evaluator_results.csv"

dense_pred = load_dataset("json", data_files=dense_pred_path)['train']
dense_metrics = pd.read_csv(dense_metrics_path)

In [18]:
display(dense_metrics)
print(dense_pred)
print(len(dense_pred['results'][0]))
print(dense_pred['results'][0])

,epoch,steps,cosine-Accuracy@1,cosine-Accuracy@3,cosine-Accuracy@5,cosine-Accuracy@10,cosine-Precision@1,cosine-Recall@1,cosine-Precision@3,cosine-Recall@3,cosine-Precision@5,cosine-Recall@5,cosine-Precision@10,cosine-Recall@10,cosine-MRR@10,cosine-NDCG@10,cosine-MAP@20
0,-1,-1,0.799107,0.933036,0.964286,0.995536,0.799107,0.799107,0.311012,0.933036,0.192857,0.964286,0.099554,0.995536,0.868548,0.899692,0.868846
1,-1,-1,0.799107,0.933036,0.964286,0.995536,0.799107,0.799107,0.311012,0.933036,0.192857,0.964286,0.099554,0.995536,0.868548,0.899692,0.868846


Dataset({
    features: ['query_id', 'query', 'results'],
    num_rows: 224
})
100
[{'corpus_id': 533, 'score': 0.7732088565826416}, {'corpus_id': 874, 'score': 0.6199048757553101}, {'corpus_id': 529, 'score': 0.5503385663032532}, {'corpus_id': 873, 'score': 0.5331900119781494}, {'corpus_id': 347, 'score': 0.5158407092094421}, {'corpus_id': 530, 'score': 0.5078306794166565}, {'corpus_id': 531, 'score': 0.5046758055686951}, {'corpus_id': 525, 'score': 0.47818058729171753}, {'corpus_id': 353, 'score': 0.4555337429046631}, {'corpus_id': 515, 'score': 0.4266102612018585}, {'corpus_id': 868, 'score': 0.4153050184249878}, {'corpus_id': 397, 'score': 0.414254754781723}, {'corpus_id': 864, 'score': 0.40650349855422974}, {'corpus_id': 532, 'score': 0.3961651027202606}, {'corpus_id': 746, 'score': 0.39209944009780884}, {'corpus_id': 870, 'score': 0.388528972864151}, {'corpus_id': 518, 'score': 0.3877118229866028}, {'corpus_id': 745, 'score': 0.3842677175998688}, {'corpus_id': 748, 'score': 0.372

In [14]:
from mine_hard_negatives import mine_bm25


bm25_pred_ids = mine_bm25(
    corpus=list(passages.values()), 
    queries_text=queries_eval.values(), 
    passage_to_id=passage_to_id, 
    mine_config={'top_k':100}
)

def evaluator_format(queries_ids, pred_ids):
    return [
        {
            "query_id": qid,
            "query": queries[qid],
            "positive": [qrels_eval[qid]],
            "documents": pids,
        }
        for qid, pids in zip(queries_ids, pred_ids)
    ]

bm25_ranking = evaluator_format(queries_eval, bm25_pred_ids)

2026-09-01 22:05:28 - bm25_mined_ids[0]: [533, 529, 525, 524, 530, 520, 872, 531, 518, 517, 482, 515, 404, 523, 526, 874, 522, 528, 873, 521, 527, 353, 480, 405, 347, 532, 519, 864, 516, 534, 777, 406, 870, 852, 868, 403, 346, 869, 30, 345, 399, 414, 558, 86, 303, 222, 557, 302, 478, 392, 204, 221, 739, 853, 452, 391, 550, 501, 553, 748, 234, 740, 662, 300, 745, 401, 426, 595, 536, 860, 38, 547, 663, 397, 747, 81, 350, 432, 428, 573, 180, 213, 560, 491, 877, 198, 396, 429, 201, 466, 461, 387, 22, 863, 509, 778, 450, 402, 101, 433]


In [ ]:
assert list(dense_pred['query_id']) == list(queries_eval.keys())
dense_pred_ids = [[p['corpus_id'] for p in ps] for ps in dense_pred['results']]
dense_ranking = evaluator_format(queries_eval, dense_pred_ids)

In [28]:
sparse_splade_pred_path = f"../result/eval/sparse_splade/Information-Retrieval_evaluation_irs_evaluator_predictions_dot.jsonl"
sparse_splade_metrics_path = f"../result/eval/sparse_splade/Information-Retrieval_evaluation_irs_evaluator_results.csv"

sparse_splade_pred = load_dataset("json", data_files=sparse_splade_pred_path)['train']
sparse_splade_metrics = pd.read_csv(sparse_splade_metrics_path)
display(sparse_splade_metrics)

,epoch,steps,dot-Accuracy@1,dot-Accuracy@3,dot-Accuracy@5,dot-Accuracy@10,dot-Precision@1,dot-Recall@1,dot-Precision@3,dot-Recall@3,dot-Precision@5,dot-Recall@5,dot-Precision@10,dot-Recall@10,dot-MRR@10,dot-NDCG@10,dot-MAP@20,dot-MAP@100,query_active_dims,query_sparsity_ratio,corpus_active_dims,corpus_sparsity_ratio,avg_flops
0,-1,-1,0.78125,0.924107,0.955357,0.986607,0.78125,0.78125,0.308036,0.924107,0.191071,0.955357,0.098661,0.986607,0.852838,0.885571,0.853136,0.853438,215.482147,0.997965,361.717865,0.996584,53.085793


In [29]:
assert list(sparse_splade_pred['query_id']) == list(queries_eval.keys())
sparse_splade_pred_ids = [[p['corpus_id'] for p in ps] for ps in sparse_splade_pred['results']]
sparse_splade_ranking = evaluator_format(queries_eval, sparse_splade_pred_ids)

In [32]:
wrrf_evaluator = WeightedReciprocalRankFusionEvaluator(
    dense_samples=dense_ranking,
    sparse_samples=bm25_ranking,
    weights=(1, 0.2),
    at_k=10,
    rrf_k=60,
    show_progress_bar=True,
    write_predictions=True,
)
wrrf_results = wrrf_evaluator()
wrrf_results

# Вывод: взвешенный гибридный rrf (dense+bm25) улучшает качество по сравнению с dense

2026-09-01 22:31:48 - ReciprocalRankFusionEvaluator: Evaluating hybrid search on the  dataset:
2026-09-01 22:31:48 - Processing 224 samples
Evaluating: 100%|██████████| 224/224 [00:00<00:00, 303.16it/s]
2026-09-01 22:31:49 - Queries: 224	Positives: Min 1.0, Mean 1.0, Max 1.0
2026-09-01 22:31:49 - ===========================================================================
2026-09-01 22:31:49 - Metric  |  Dense   |  Sparse  |  Fusion  | Gain vs Dense | Gain vs Sparse |
2026-09-01 22:31:49 - ---------------------------------------------------------------------------
2026-09-01 22:31:49 - MAP     |   86.88% |   80.33% |   88.29% |        +1.41% |         +7.96% |
2026-09-01 22:31:49 - MRR@10  |   86.85% |   80.03% |   88.29% |        +1.44% |         +8.27% |
2026-09-01 22:31:49 - NDCG@10 |   89.97% |   83.19% |   91.18% |        +1.21% |         +7.99% |
2026-09-01 22:31:49 - ===========================================================================


{'dense_map': 0.8688456632653061,
 'dense_mrr@10': 0.868548044217687,
 'dense_ndcg@10': 0.8996919151844279,
 'sparse_map': 0.8033258734131417,
 'sparse_mrr@10': 0.800283446712018,
 'sparse_ndcg@10': 0.8318894850866206,
 'map': 0.8829471371882087,
 'mrr@10': 0.8829471371882087,
 'ndcg@10': 0.9117643628177718}

In [33]:
wrrf_evaluator = WeightedReciprocalRankFusionEvaluator(
    dense_samples=dense_ranking,
    sparse_samples=sparse_splade_ranking,
    weights=(1, 0.25),
    at_k=10,
    rrf_k=60,
    show_progress_bar=True,
    write_predictions=True,
)
wrrf_results = wrrf_evaluator()
wrrf_results

# Вывод: взвешенный гибридный rrf (dense+splade) еще сильнее улучшает качество по сравнению с (dense+bm25)

2026-09-01 22:32:20 - ReciprocalRankFusionEvaluator: Evaluating hybrid search on the  dataset:
2026-09-01 22:32:20 - Processing 224 samples
Evaluating: 100%|██████████| 224/224 [00:00<00:00, 282.07it/s]
2026-09-01 22:32:21 - Queries: 224	Positives: Min 1.0, Mean 1.0, Max 1.0
2026-09-01 22:32:21 - ===========================================================================
2026-09-01 22:32:21 - Metric  |  Dense   |  Sparse  |  Fusion  | Gain vs Dense | Gain vs Sparse |
2026-09-01 22:32:21 - ---------------------------------------------------------------------------
2026-09-01 22:32:21 - MAP     |   86.88% |   85.34% |   88.81% |        +1.92% |         +3.46% |
2026-09-01 22:32:21 - MRR@10  |   86.85% |   85.28% |   88.81% |        +1.95% |         +3.52% |
2026-09-01 22:32:21 - NDCG@10 |   89.97% |   88.56% |   91.60% |        +1.63% |         +3.04% |
2026-09-01 22:32:21 - ===========================================================================


{'dense_map': 0.8688456632653061,
 'dense_mrr@10': 0.868548044217687,
 'dense_ndcg@10': 0.8996919151844279,
 'sparse_map': 0.8534382086167801,
 'sparse_mrr@10': 0.8528380102040816,
 'sparse_ndcg@10': 0.8855710842974792,
 'map': 0.8880580357142858,
 'mrr@10': 0.8880580357142858,
 'ndcg@10': 0.9160096804512827}

#
#### Pipeline: retriever (dense) → reranker

In [ ]:
from hybrid_retrieval.eval import eval_pipeline

In [ ]:
model_name = "i1j/reranker"

model_ce = CrossEncoder(
    model_name, 
    device='cuda' if torch.cuda.is_available() else 'cpu'
)

In [ ]:
eval_pipeline(model_ce, dense_pred, qrels_eval, passages, top_k_retrieval=30, map_at_k=20)

{'accuracy@1': 0.8973214285714286,
 'accuracy@10': 1.0,
 'accuracy@3': 0.9821428571428571,
 'accuracy@5': 0.9955357142857143,
 'map@20': 0.9406249999999999,
 'mrr@10': 0.9406249999999999,
 'ndcg@1': 0.8973214285714286,
 'ndcg@10': 0.9556623188127732,
 'ndcg@3': 0.9484997602838029,
 'ndcg@5': 0.9540721081560409,
 'precision@1': 0.8973214285714286,
 'precision@10': 0.1,
 'precision@3': 0.3273809523809524,
 'precision@5': 0.1991071428571429,
 'recall@1': 0.8973214285714286,
 'recall@10': 1.0,
 'recall@3': 0.9821428571428571,
 'recall@5': 0.9955357142857143}

#
#### Experiments

In [ ]:
# --- Experiments ------------------------------------------------------------

# dense:
#
# best: hard negatives=2 min=5 n=30 mp=fp16 B=16 batch_sampler=NO_DUPLICATES
# Without Gains:
#   +1 bm25 hard negative
#   batch_sampler=custom
#   hard negatives=4 
#   B=8
#   hard negatives min=2 n=20

# sparse SPLADE:
#
# Так как ru splade-моделей нет то варианты:
# 1. единственная мультиязычная от OpenSearch opensearch-project/opensearch-neural-sparse-encoding-multilingual-v1. 
#    На MIRACL по русскому она даёт NDCG@10 = 0.658 против 0.256 у BM25.
#    Но модель асимметричная и inference-free — документы кодируются энкодером, а для запросов используется только 
#    токенизатор и таблица весов (IDF). То есть расширения запроса там нет.
#
# 2. брать en splade-модель: splade очень сильно зависит от получаемых токенов.
#    из-за того что словарь en то русские тексты будут в основном токенизироваться на буквы которые не будут иметь смысла
#    в pooling-слое, представления запроса и документа будут просто одинаковыми буквами vocab - при max-пулинге они все насыщаются, 
#    скалярное произведение перестаёт различать документы, и ранжирование вырождается.
#
# 3. брать ru энкодер с MLM-головой: энкодер не обучен ни под IR ни под splade, нужно очень много данных для обучения.
#    в случае если у HF модели MLM-головы еще и нет, но она будет случайной инициализированной. 